# Module `graph_generator.py`

Ce notebook explique comment une instance CESIPATH est construite pas a pas.

L'objectif est de bien distinguer :

1. le graphe de base ;
2. le graphe residuel apres contraintes statiques ;
3. le graphe complete par Dijkstra ;
4. les snapshots dynamiques recalcules a chaque tour.

In [ ]:
import sys
from pathlib import Path
from pprint import pprint

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.metric_closure import check_triangle_inequality
from cesipath.models import GraphGenerationConfig

## Parametres de generation

Ici, `auto_density_profile=True` laisse le projet choisir une densite adaptee au nombre de sommets.

Les contraintes statiques sont :

- `forbidden_rate` : chance qu'une vraie route soit interdite statiquement ;
- `surcharge_rate` : chance qu'une vraie route ait un surcout statique ;
- `vehicle_capacity` : capacite du vehicule, utile pour les futures tournees.

La dynamique est configuree, mais elle ne s'applique pas pendant la creation du graphe statique. Elle intervient seulement ensuite, dans les snapshots dynamiques.

In [ ]:
config = GraphGenerationConfig(
    node_count=7,
    seed=42,
    edge_density=None,
    auto_density_profile=True,
    forbidden_rate=0.1,
    surcharge_rate=0.25,
    vehicle_capacity=40,
    dynamic_forbid_probability=0.03,
    dynamic_restore_probability=0.2,
    dynamic_max_disabled_ratio=0.2,
)

generator = GraphGenerator(config)
instance = generator.generate()

pprint(instance.summary())

## Lecture du resume

Le resume confirme que l'instance respecte les contraintes de qualite :

- `base_density` : densite du graphe de base ;
- `residual_density` : densite apres interdictions statiques ;
- `residual_average_degree` : degre moyen apres contraintes ;
- `forbidden_edge_count` : nombre de routes interdites statiquement ;
- `surcharged_edge_count` : nombre de routes surchargees statiquement.

Si une instance sort des bornes, le generateur la rejette et recommence jusqu'a `generation_max_attempts`.

In [ ]:
print("Demandes :", instance.demands)
print("Coordonnees :")
pprint(instance.coordinates)
print("Densite cible base :", config.resolved_edge_density)
print("Densite base :", instance.base_density)
print("Densite residuelle :", instance.residual_density)
print("Degre moyen residuel :", instance.residual_average_degree)

## Les trois matrices importantes

`base_costs` represente le graphe physique initial.

`residual_costs` represente le graphe apres application des contraintes statiques. Les routes interdites disparaissent de cette matrice.

`completed_costs` represente le graphe complet obtenu par Dijkstra sur le graphe residuel. C'est cette fermeture metrique qui permet de rester coherent avec une logique de TSP metrique.

In [ ]:
def print_matrix(title, matrix):
    print(title)
    for row in matrix:
        print(row)
    print()

print_matrix("Matrice du graphe de base :", instance.base_costs)
print_matrix("Matrice du graphe residuel :", instance.residual_costs)
print_matrix("Matrice complete par Dijkstra :", instance.completed_costs)

## Verification de l'inegalite triangulaire

La verification est importee depuis `metric_closure.py`, car ce n'est plus le role du generateur de connaitre les details mathematiques de Dijkstra.

Comme la matrice complete est construite par plus courts chemins, elle doit respecter :

```text
d(i, j) <= d(i, k) + d(k, j)
```

Cette verification est importante car elle confirme que la matrice utilisee pour la resolution reste bien dans un cadre metrique.

In [ ]:
ok, violation = check_triangle_inequality(instance.completed_costs)
print("Inegalite triangulaire respectee ?", ok)
print("Violation :", violation)
print("Chemin reel associe a 0 -> 3 :", instance.completed_paths.get((0, 3)))

## Passage a la dynamique

La dynamique est maintenant geree par `DynamicNetworkSimulator`, pas directement par `GraphGenerator`.

Un `DynamicGraphSnapshot` represente l'etat du reseau a un tour donne.

A chaque appel a `simulator.advance(...)` :

1. les couts des vraies routes evoluent ;
2. certaines routes peuvent passer temporairement `OFF` ;
3. le code verifie connexite, densite active, degre moyen actif et ratio maximum de routes `OFF` ;
4. Dijkstra est relance pour reconstruire la matrice complete courante.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.initialize_snapshot()
print("Tour initial :", snapshot.step)
print("Aretes actives :", snapshot.active_edge_count)

next_snapshot = simulator.advance(snapshot)
print("Tour suivant :", next_snapshot.step)
print("Aretes actives :", next_snapshot.active_edge_count)
print_matrix("Matrice complete dynamique apres un tour :", next_snapshot.completed_costs)